In [0]:
%pip install deltalake

In [0]:
import os
import io
import time
import json
import logging
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

# ─────────────────────────────────────────────
# CONFIGURAÇÃO DE LOGS
# ─────────────────────────────────────────────
logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger("squad2")

# ─────────────────────────────────────────────
# CARREGAMENTO DE CREDENCIAIS
# ─────────────────────────────────────────────
load_dotenv()

# ADLS
ADLS_CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
ADLS_TENANT_ID       = os.getenv("ADLS_TENANT_ID")
ADLS_CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
ADLS_STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT")
ADLS_CONTAINER       = os.getenv("ADLS_CONTAINER")
SQUAD2_CONTAINER     = os.getenv("SQUAD2_CONTAINER", "squad2")

# SQL Server
SQL_HOST     = os.getenv("SQL_HOST")
SQL_DATABASE = os.getenv("SQL_DATABASE")
SQL_USERNAME = os.getenv("SQL_USERNAME")
SQL_PASSWORD = os.getenv("SQL_PASSWORD")

# Opções SQL reutilizáveis
SQL_OPTIONS = {
    "host"    : SQL_HOST,
    "database": SQL_DATABASE,
    "user"    : SQL_USERNAME,
    "password": SQL_PASSWORD
}

# ─────────────────────────────────────────────
# CONFIGURAÇÕES DO PROJETO
# ─────────────────────────────────────────────

# Caminhos Medalhão
PATHS = {
    "raw"        : "real-time-data",
    "bronze"     : "squad2/bronze",
    "silver"     : "squad2/silver",
    "gold"       : "squad2/gold",
    "checkpoint" : "squad2/checkpoints"
}

# Tabelas sob responsabilidade do Squad 2
TABELAS_SQUAD2 = [
    "ecommerce_categorias",
    "ecommerce_itens_pedido",
    "ecommerce_produtos",
    "ecommerce_pedidos"
]

# Schema destino SQL Server
SQL_SCHEMA = "squad2"
SQL_PREFIX = ""

# ─────────────────────────────────────────────
# FUNÇÕES — ADLS
# ─────────────────────────────────────────────

def get_adls_client() -> DataLakeServiceClient:
    """Cria e retorna um cliente autenticado do ADLS Gen2 via Service Principal."""
    credential = ClientSecretCredential(
        tenant_id     = ADLS_TENANT_ID,
        client_id     = ADLS_CLIENT_ID,
        client_secret = ADLS_CLIENT_SECRET
    )
    return DataLakeServiceClient(
        account_url = f"https://{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        credential  = credential
    )

def get_container_client():
    """Retorna o cliente do container configurado."""
    return get_adls_client().get_file_system_client(ADLS_CONTAINER)

def get_squad2_client():
    """Retorna o cliente do container squad2 para escrita de dados."""
    return get_adls_client().get_file_system_client(SQUAD2_CONTAINER)

def listar_snapshots(base_path: str = None) -> set:
    """Lista todas as pastas de snapshot disponíveis no lake."""
    if base_path is None:
        base_path = PATHS["raw"]

    snapshots        = set()
    container_client = get_container_client()
    paths            = container_client.get_paths(path=base_path, recursive=True)

    for item in paths:
        partes = item.name.replace(base_path + "/", "").split("/")
        if len(partes) == 4 and item.is_directory:
            snapshots.add("/".join(partes))

    return snapshots

def ler_parquet(snapshot_id: str, tabela: str) -> "pyspark.sql.DataFrame":
    """Lê um arquivo parquet de um snapshot específico do ADLS."""
    base_path        = PATHS["raw"]
    file_path        = f"{base_path}/{snapshot_id}/{tabela}.parquet"
    container_client = get_container_client()
    file_client      = container_client.get_file_client(file_path)

    bytes_data = file_client.download_file().readall()
    pdf        = pd.read_parquet(io.BytesIO(bytes_data))

    return spark.createDataFrame(pdf)

# ─────────────────────────────────────────────
# FUNÇÕES — SQL SERVER
# ─────────────────────────────────────────────

def get_destino_sql(tabela: str) -> str:
    return f"{SQL_SCHEMA}.{SQL_PREFIX}{tabela}"

def gravar_sql(df: "pyspark.sql.DataFrame", tabela: str, mode: str = "overwrite") -> bool:
    destino = get_destino_sql(tabela)
    try:
        df.write \
            .format("sqlserver") \
            .options(**SQL_OPTIONS) \
            .option("dbtable", destino) \
            .mode(mode) \
            .save()
        log.info(f"Gravado: {destino} → {df.count()} linhas")
        return True
    except Exception as e:
        log.error(f"Erro ao gravar {destino}: {str(e)}")
        return False

def ler_sql(tabela: str) -> "pyspark.sql.DataFrame":
    destino = get_destino_sql(tabela)
    return spark.read \
        .format("sqlserver") \
        .options(**SQL_OPTIONS) \
        .option("dbtable", destino) \
        .load()

def validar_gravacao(tabela: str) -> bool:
    try:
        df    = ler_sql(tabela)
        count = df.count()
        log.info(f"Validado: {get_destino_sql(tabela)} → {count} linhas")
        return count > 0
    except Exception as e:
        log.error(f"Erro ao validar {tabela}: {str(e)}")
        return False

# ─────────────────────────────────────────────
# FUNÇÕES — DELTA LAKE (ADLS via deltalake-python)
# ─────────────────────────────────────────────

def get_delta_path(camada: str, tabela: str) -> str:
    """Retorna a URI nativa exigida pelo storage_options do deltalake-python."""
    return f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/{camada}/{tabela}"

def get_storage_options() -> dict:
    return {
        "account_name"  : ADLS_STORAGE_ACCOUNT,
        "tenant_id"     : ADLS_TENANT_ID,
        "client_id"     : ADLS_CLIENT_ID,
        "client_secret" : ADLS_CLIENT_SECRET
    }

def delta_existe(camada: str, tabela: str) -> bool:
    try:
        from deltalake import DeltaTable
        DeltaTable(get_delta_path(camada, tabela), storage_options=get_storage_options())
        return True
    except Exception:
        return False

def gravar_delta(df: "pyspark.sql.DataFrame", camada: str, tabela: str, mode: str = "append", particionar: bool = True) -> bool:
    import pyarrow as pa
    from deltalake.writer import write_deltalake

    path         = get_delta_path(camada, tabela)
    storage_opts = get_storage_options()
    modo_real    = mode if delta_existe(camada, tabela) else "overwrite"

    try:
        #pdf = df.toPandas()
        pdf = pd.DataFrame(df.collect(), columns=df.columns)
        
        # Correção de fuso horário
        for col_name in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col_name]):
                pdf[col_name] = pdf[col_name].dt.tz_localize(None)
                
        tabela_arrow = pa.Table.from_pandas(pdf)

        partition_by = None
        if particionar and camada == "bronze":
            partition_by = ["ingestion_year", "ingestion_month", "ingestion_day", "ingestion_hour"]
            colunas = pdf.columns.tolist()
            if not all(c in colunas for c in partition_by):
                partition_by = None

        write_deltalake(
            table_or_uri    = path,
            data            = tabela_arrow,
            mode            = modo_real,
            storage_options = storage_opts,
            partition_by    = partition_by
        )
        log.info(f"Gravado com sucesso: {path} → {len(pdf)} linhas. partições: {partition_by}")
        return True
    except Exception as e:
        if "does not match table partitioning" in str(e):
            try:
                log.warning("Forçando alinhamento de partições no Storage...")
                write_deltalake(
                    table_or_uri=path, data=tabela_arrow, mode="overwrite",
                    storage_options=storage_opts, partition_by=partition_by, schema_mode="overwrite"
                )
                return True
            except Exception as e_inner:
                log.error(f"Falha ao reestruturar esquema Delta: {str(e_inner)}")
                return False
        log.error(f"Erro ao gravar {path}: {str(e)}")
        return False

def ler_delta(camada: str, tabela: str) -> "pyspark.sql.DataFrame":
    from deltalake import DeltaTable
    import pyarrow as pa

    path         = get_delta_path(camada, tabela)
    storage_opts = get_storage_options()

    try:
        dt = DeltaTable(path, storage_options=storage_opts)
        arrow_table = dt.to_pyarrow_table().combine_chunks()
        pdf = arrow_table.to_pandas()
        return spark.createDataFrame(pdf)
    except Exception as e1:
        log.warning(f"Engine principal falhou, executando Fallback via Azure SDK...")
        squad2_client = get_squad2_client()
        frames = []
        paths = list(squad2_client.get_paths(path=f"{camada}/{tabela}", recursive=True))
        parquets = [p.name for p in paths if p.name.endswith(".parquet") and "_delta_log" not in p.name]

        for p in parquets:
            file_client = squad2_client.get_file_client(p)
            bytes_data  = file_client.download_file().readall()
            pdf_part    = pd.read_parquet(io.BytesIO(bytes_data))
            frames.append(pdf_part)

        if frames:
            return spark.createDataFrame(pd.concat(frames, ignore_index=True))
        else:
            raise Exception(f"Nenhum arquivo encontrado em {camada}/{tabela}")

# ── Leitura Delta Silver ───────────────────────────────────────────────────────
def ler_delta_silver(nome_tabela: str):
    """
    Lê uma tabela Delta da camada Silver do Squad 2.

    Args:
        nome_tabela: nome da tabela (ex: 'ecommerce_categorias')

    Returns:
        Spark DataFrame com os dados da Silver
    """
    from pyspark.sql import SparkSession
    import io

    spark = SparkSession.builder.getOrCreate()

    container_client = get_squad2_client()
    path_silver      = f"silver/{nome_tabela}"

    log.info(f"Lendo Silver: {nome_tabela}")

    # Lista todos os arquivos parquet da pasta silver
    paths = [
        p.name
        for p in container_client.get_paths(path=path_silver, recursive=True)
        if p.name.endswith(".parquet") and "_delta_log" not in p.name
    ]

    if not paths:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em {path_silver}")

    log.info(f"{len(paths)} arquivo(s) parquet encontrado(s)")

    # Lê e concatena todos os arquivos
    dfs = []
    for file_path in paths:
        file_client = container_client.get_file_client(file_path)
        data        = file_client.download_file().readall()
        df_pd       = pd.read_parquet(io.BytesIO(data))
        dfs.append(df_pd)

    df_final = pd.concat(dfs, ignore_index=True)
    log.info(f"Total lido: {len(df_final):,} registros")

    return spark.createDataFrame(df_final)

print(" ler_delta_silver carregada.")

def get_nome_delta(camada: str, tabela: str) -> str:
    return get_delta_path(camada, tabela)


# ─────────────────────────────────────────────
# FUNÇÕES — QUALIDADE & OBSERVABILIDADE
# ─────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

CORES_TIPO = {
    "Completude"  : "#E74C3C",
    "Unicidade"   : "#E67E22",
    "Domínio"     : "#F1C40F",
    "Formato"     : "#3498DB",
    "Referencial" : "#9B59B6",
    "Negócio"     : "#1ABC9C",
    "Consistência": "#95A5A6",
}

def plot_erros(resultados: list, titulo: str) -> None:
    """
    Gera gráfico de barras horizontais com quantidade de erros por regra.

    Args:
        resultados : lista de dicts com chaves:
                     codigo, descricao, tipo, erro_count, total
        titulo     : nome da tabela para o título do gráfico
    """
    if not resultados:
        print("  Nenhum resultado para plotar.")
        return

    df_plot = pd.DataFrame(resultados)
    df_plot["pct"]   = (df_plot["erro_count"] / df_plot["total"] * 100).round(2)
    df_plot["label"] = df_plot["codigo"] + " — " + df_plot["descricao"]
    df_plot["cor"]   = df_plot["tipo"].map(CORES_TIPO).fillna("#BDC3C7")
    df_plot          = df_plot.sort_values("erro_count", ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(4, len(df_plot) * 0.7)))

    bars = ax.barh(
        df_plot["label"],
        df_plot["erro_count"],
        color=df_plot["cor"],
        edgecolor="white",
        height=0.6,
    )

    for bar, (_, row) in zip(bars, df_plot.iterrows()):
        ax.text(
            bar.get_width() + max(df_plot["erro_count"].max() * 0.01, 0.1),
            bar.get_y() + bar.get_height() / 2,
            f"{int(bar.get_width()):,}  ({row['pct']}%)",
            va="center", fontsize=9, color="#2C3E50",
        )

    legend_patches = [
        mpatches.Patch(color=cor, label=tipo)
        for tipo, cor in CORES_TIPO.items()
        if tipo in df_plot["tipo"].values
    ]
    ax.legend(
        handles=legend_patches,
        title="Tipo de problema",
        loc="lower right",
        fontsize=8,
    )

    ax.set_title(f"Erros de Qualidade — {titulo}", fontsize=13, fontweight="bold", pad=14)
    ax.set_xlabel("Quantidade de registros com erro", fontsize=10)

    max_val = df_plot["erro_count"].max()
    ax.set_xlim(0, max_val * 1.18 if max_val > 0 else 10)

    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

    print(f"\n{'='*60}")
    print(f"  RESUMO — {titulo}")
    print(f"{'='*60}")
    for _, row in df_plot.sort_values("erro_count", ascending=False).iterrows():
        status = " FAIL" if row["erro_count"] > 0 else " PASS"
        print(f"  {status}  {row['codigo']:<10} {row['erro_count']:>6,} erros  ({row['pct']}%)")
    print(f"{'='*60}")

print(" plot_erros carregada.")


# ─────────────────────────────────────────────
# FUNÇÕES — CHECKPOINT & CONTROLE (JSON)
# ─────────────────────────────────────────────

def get_control_path(camada: str, tabela: str) -> tuple:
    """Retorna o cliente do container e o caminho do arquivo de controle JSON.

    Padrão: squad2/control/<camada>/<tabela>/control_file.json
    """
    container_client = get_squad2_client()
    control_file     = f"control/{camada}/{tabela}/control_file.json"
    return container_client, control_file

def ler_checkpoint(camada: str, tabela: str) -> set:
    """Lê o arquivo JSON de controle e retorna o set de snapshots já processados.

    Se o arquivo não existir, retorna set vazio (inicia carga limpa sem erro).
    """
    container_client, control_file = get_control_path(camada, tabela)
    processados = set()
    try:
        file_client = container_client.get_file_client(control_file)
        conteudo    = file_client.download_file().readall().decode("utf-8")
        dados       = json.loads(conteudo)
        processados = set(dados.get("processed_snapshots", []))
        log.info(f"{len(processados)} snapshot(s) recuperados do JSON de controle.")
    except Exception:
        log.info(f"Nenhum controle JSON localizado em {control_file}. Iniciando carga limpa.")
    return processados

def salvar_checkpoint(camada: str, tabela: str, processados: set, status: str = "CONCLUIDO") -> None:
    """
    Persiste o set atualizado de snapshots processados no arquivo JSON de controle.
    Inclui campo status para coordenação entre camadas.

    Args:
        camada     : camada do pipeline ('bronze', 'silver', 'gold')
        tabela     : nome da tabela
        processados: set de snapshot_ids já processados
        status     : 'PROCESSANDO' durante execução, 'CONCLUIDO' ao finalizar
    """
    container_client, control_file = get_control_path(camada, tabela)
    file_client = container_client.get_file_client(control_file)

    dados_controle = {
        "tabela"                      : tabela,
        "camada"                      : camada,
        "status"                      : status,
        "ultima_atualizacao"          : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_snapshots_processados" : len(processados),
        "processed_snapshots"         : sorted(list(processados))
    }
    conteudo = json.dumps(dados_controle, indent=4).encode("utf-8")
    try:
        file_client.get_file_properties()
        file_client.upload_data(conteudo, overwrite=True)
    except Exception:
        file_client.create_file()
        file_client.upload_data(conteudo, overwrite=True)
    log.info(f"Checkpoint [{status}] salvo em {control_file}")

print("Salvar_checkpoint atualizado.")

def ler_status_checkpoint(camada: str, tabela: str) -> str:
    """
    Lê o status atual do control file de uma camada/tabela.

    Returns:
        'CONCLUIDO', 'PROCESSANDO' ou 'PENDENTE' (se não existir)
    """
    container_client, control_file = get_control_path(camada, tabela)
    try:
        file_client = container_client.get_file_client(control_file)
        conteudo    = file_client.download_file().readall().decode("utf-8")
        dados       = json.loads(conteudo)
        return dados.get("status", "CONCLUIDO")
    except Exception:
        return "PENDENTE"


def camada_anterior_concluida(camada_atual: str, tabelas: list) -> bool:
    """
    Verifica se TODAS as tabelas da camada anterior estão com status CONCLUIDO.
    Usada pelo polling_loop para garantir dependência entre camadas.

    Args:
        camada_atual: camada que quer iniciar ('silver' ou 'gold')
        tabelas     : lista de tabelas a verificar na camada anterior

    Returns:
        True se todas as tabelas da camada anterior estão CONCLUIDO
    """
    CAMADA_ANTERIOR = {
        "silver": "bronze",
        "gold"  : "silver",
    }

    camada_dep = CAMADA_ANTERIOR.get(camada_atual)
    if not camada_dep:
        return True  # Bronze não tem dependência

    log.info(f"Verificando dependência: {camada_dep.upper()} → {camada_atual.upper()}")

    for tabela in tabelas:
        status = ler_status_checkpoint(camada_dep, tabela)
        log.info(f"  {camada_dep}/{tabela}: {status}")

        if status != "CONCLUIDO":
            log.warning(f"  ⏳ Aguardando {camada_dep}/{tabela} concluir...")
            return False

    log.info(f"Todas as tabelas da {camada_dep.upper()} estão CONCLUIDO — {camada_atual.upper()} pode iniciar.")
    return True

print("Ler_status_checkpoint e camada_anterior_concluida carregadas.")

# ─────────────────────────────────────────────
# FUNÇÕES — UTILITÁRIAS
# ─────────────────────────────────────────────

def get_snapshot_mais_recente(base_path: str = None) -> str:
    snapshots = listar_snapshots(base_path)
    if not snapshots:
        raise ValueError("Nenhum snapshot encontrado.")
    return sorted(snapshots)[-1]

def log_inicio(notebook: str) -> datetime:
    inicio = datetime.now()
    log.info(f"{'='*50}\nINÍCIO: {notebook}\nData  : {inicio.strftime('%Y-%m-%d %H:%M:%S')}\n{'='*50}")
    return inicio

def log_fim(notebook: str, inicio: datetime) -> None:
    fim      = datetime.now()
    duracao  = (fim - inicio).seconds
    log.info(f"{'='*50}\nFIM   : {notebook}\nTempo : {duracao}s\n{'='*50}")

def _validar_credenciais() -> None:
    credenciais = {
        "ADLS_CLIENT_ID"      : ADLS_CLIENT_ID,
        "ADLS_TENANT_ID"      : ADLS_TENANT_ID,
        "ADLS_CLIENT_SECRET"  : ADLS_CLIENT_SECRET,
        "ADLS_STORAGE_ACCOUNT": ADLS_STORAGE_ACCOUNT,
        "ADLS_CONTAINER"      : ADLS_CONTAINER,
        "SQL_HOST"            : SQL_HOST,
        "SQL_DATABASE"        : SQL_DATABASE
    }
    for nome, valor in credenciais.items():
        if not valor:
            raise EnvironmentError(f"Credencial ausente: {nome}")
    log.info("Helpers carregados! Todas as credenciais OK.")


# ─────────────────────────────────────────────
# FUNÇÕES — POLLING LOOP
# ─────────────────────────────────────────────

def polling_loop(
    processar_fn : object,
    camada       : str,
    tabela       : str,
    intervalo_s  : int  = 30,
    max_ciclos   : int  = None,
    tabelas_dep  : list = None
) -> None:
    """
    Executa pipeline contínuo verificando a cada intervalo_s segundos
    se há novos snapshots e se a camada anterior está concluída.

    Args:
        processar_fn : função que processa um snapshot → bool
        camada       : camada atual ('bronze', 'silver', 'gold')
        tabela       : nome da tabela
        intervalo_s  : segundos entre ciclos (padrão: 30)
        max_ciclos   : limite de ciclos — None = infinito
        tabelas_dep  : tabelas da camada anterior a verificar
                       (padrão: TABELAS_SQUAD2)
    """
    if tabelas_dep is None:
        tabelas_dep = TABELAS_SQUAD2

    log.info(f"{'='*55}")
    log.info(f"  POLLING LOOP INICIADO")
    log.info(f"  Camada   : {camada.upper()}")
    log.info(f"  Tabela   : {tabela}")
    log.info(f"  Intervalo: {intervalo_s}s")
    log.info(f"{'='*55}")

    ciclo = 0

    while True:
        ciclo += 1
        agora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log.info(f"[Ciclo {ciclo}] {agora}")

        try:
            # 1. Verifica se camada anterior está concluída
            if not camada_anterior_concluida(camada, tabelas_dep):
                log.info(f"[Ciclo {ciclo}] ⏳ Camada anterior pendente — aguardando {intervalo_s}s...")
            else:
                # 2. Lê snapshots disponíveis
                snapshots_disponiveis = sorted(listar_snapshots())
                processados           = ler_checkpoint(camada, tabela)
                novos                 = [s for s in snapshots_disponiveis if s not in processados]

                if not novos:
                    log.info(f"[Ciclo {ciclo}] ✅ Em dia — nenhum snapshot novo. Aguardando {intervalo_s}s...")
                else:
                    log.info(f"[Ciclo {ciclo}] 🆕 {len(novos)} snapshot(s) novo(s): {novos}")

                    # Marca como PROCESSANDO antes de iniciar
                    salvar_checkpoint(camada, tabela, processados, status="PROCESSANDO")

                    for snapshot_id in novos:
                        log.info(f"[Ciclo {ciclo}] ▶ Processando: {snapshot_id}")
                        try:
                            sucesso = processar_fn(snapshot_id)

                            if sucesso:
                                processados.add(snapshot_id)
                                log.info(f"[Ciclo {ciclo}]  {snapshot_id} processado.")
                            else:
                                log.warning(f"[Ciclo {ciclo}]  {snapshot_id} retornou False — retentado no próximo ciclo.")

                        except Exception as e_inner:
                            log.error(f"[Ciclo {ciclo}]  Erro em {snapshot_id}: {str(e_inner)}")
                            log.warning(f"[Ciclo {ciclo}] Continuando — será retentado.")

                    # Marca como CONCLUIDO ao finalizar todos os snapshots
                    salvar_checkpoint(camada, tabela, processados, status="CONCLUIDO")
                    log.info(f"[Ciclo {ciclo}] 🏁 {tabela} marcada como CONCLUIDO.")

        except Exception as e_outer:
            log.error(f"[Ciclo {ciclo}]  Erro no ciclo: {str(e_outer)}")
            log.warning(f"[Ciclo {ciclo}] Tentando novamente em {intervalo_s}s...")

        # 3. Verifica limite de ciclos
        if max_ciclos and ciclo >= max_ciclos:
            log.info(f"[Ciclo {ciclo}] Limite de {max_ciclos} ciclo(s) atingido — encerrando.")
            break

        time.sleep(intervalo_s)

print("Polling_loop atualizado.")

_validar_credenciais()

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp, year, month, dayofmonth, hour

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA = "ecommerce_pedidos"
CAMADA = "bronze"

inicio = log_inicio("feat_squad2_" + CAMADA + "_" + TABELA)
log.info("Tabela : " + TABELA)
log.info("Camada : " + CAMADA)
log.info("Path   : " + get_delta_path(CAMADA, TABELA))

In [0]:
def processar_snapshot(snapshot_id: str) -> bool:
    try:
        # 1. Lê o dataframe
        df = ler_parquet(snapshot_id, TABELA)

        # 2. MOSTRA OS DADOS AQUI (Coloque logo após a leitura)
        df.show(10, truncate=False)

        source_file = "real-time-data/" + snapshot_id + "/" + TABELA + ".parquet"

        df_bronze = df \
            .withColumn("bronze_source_file", lit(source_file)) \
            .withColumn("bronze_ingested_at", current_timestamp()) \
            .withColumn("_source",            lit("real-time-data")) \
            .withColumn("_camada",            lit(CAMADA)) \
            .withColumn("ingestion_year",     year(current_timestamp()).cast("string")) \
            .withColumn("ingestion_month",    month(current_timestamp()).cast("string")) \
            .withColumn("ingestion_day",      dayofmonth(current_timestamp()).cast("string")) \
            .withColumn("ingestion_hour",     hour(current_timestamp()).cast("string"))

        sucesso = gravar_delta(df_bronze, CAMADA, TABELA)

        if sucesso:
            log.info("OK " + snapshot_id + " -> " + str(df_bronze.count()) + " linhas gravadas.")

        return sucesso

    except Exception as e:
        log.error("Erro ao processar " + snapshot_id + ": " + str(e))
        return False
    


In [0]:
try:
    # 1. Identifica o snapshot mais recente no Lake
    ultimo_snapshot = get_snapshot_mais_recente()
    log.info(f"Carregando visualização interativa do snapshot: {ultimo_snapshot}")

    # 2. Carrega o DataFrame do Spark
    df_interativo = ler_parquet(ultimo_snapshot, TABELA)

    # 3. Renderiza a tabela interativa na interface do Notebook
    display(df_interativo)

except Exception as e:
    log.error(f"Erro ao executar o display(): {str(e)}")